# PSA Translator — Load from a Shared Drive Folder & Translate

A standalone notebook that **downloads the already-trained models from a fixed shared Google Drive
folder** and translates any English PSA into **Kiswahili** and **Ekegusii**. Anyone can run it — the
models come from the shared folder, not from the runner's own Drive. No training, no dataset needed.

- **Kiswahili** → **NLLB-200** (strongest for English↔Kiswahili; downloads once, no checkpoint needed).
- **Ekegusii** → your fine-tuned **mT5** checkpoint pulled from the shared folder.

> **Important:** for others to access it, the shared folder must be set to
> **"Anyone with the link → Viewer"** in Google Drive.

Run cells top to bottom, then use cell **4** (one sentence) or **5** (keep translating).

## 0 · Install dependencies

In [ ]:
import importlib.util, subprocess, sys
REQ = {"transformers": "transformers>=4.41", "sentencepiece": "sentencepiece",
       "accelerate": "accelerate>=0.30", "gdown": "gdown"}
missing = [spec for mod, spec in REQ.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Done. If Colab asks to restart, restart and re-run this cell.")
else:
    print("All dependencies already present.")

## 1 · Imports

In [ ]:
import re
from pathlib import Path
from functools import lru_cache
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| device:", DEVICE)

## 2 · Download the trained models from the shared Drive folder

This pulls the models from **one fixed shared folder**, so anyone running the notebook gets the same
models without needing them in their own Drive. (The folder must be shared "Anyone with the link".)

In [ ]:
import gdown

# Fixed shared Drive folder holding the trained models. Change only if you move the folder.
SHARED_FOLDER_URL = "https://drive.google.com/drive/folders/1fEIXJNYa0Rjv8M2S5Cb3rzIx2gj9h0do"
DL_DIR = Path("shared_models")

if not DL_DIR.exists() or not any(DL_DIR.rglob("*")):
    print("Downloading models from the shared Drive folder (this can take a few minutes)…")
    try:
        gdown.download_folder(SHARED_FOLDER_URL, output=str(DL_DIR),
                              quiet=False, use_cookies=False, remaining_ok=True)
    except Exception as e:
        print("Download failed:", e)
        print("Make sure the folder is shared as 'Anyone with the link -> Viewer'.")
else:
    print("Already downloaded to", DL_DIR)

# Robustly locate the mT5 checkpoint folder(s) wherever they landed in the download.
mt5_dirs = [p for p in DL_DIR.rglob("mt5_*") if p.is_dir()]
MODEL_DIR = mt5_dirs[0].parent if mt5_dirs else DL_DIR
print("\nUsing MODEL_DIR:", MODEL_DIR)
print("Checkpoints found:", sorted(p.name for p in MODEL_DIR.glob("*") if p.is_dir()) or "(none)")

## 3 · Configuration & translation function

In [ ]:
NLLB_NAME = "facebook/nllb-200-distilled-600M"
NLLB_CODE = {"English": "eng_Latn", "Kiswahili": "swh_Latn", "Ekegusii": None}
NUM_BEAMS = 4            # beam search for nicer output (lower to 1 for speed)
MAX_LEN   = 128

# Which model handles each target language.
# To use a fine-tuned mT5 for Kiswahili instead of NLLB (if that checkpoint is in the folder):
#   "Kiswahili": (MODEL_DIR / "mt5_English_to_Kiswahili", "mt5")
TARGETS = {
    "Kiswahili": (NLLB_NAME, "nllb"),
    "Ekegusii":  (MODEL_DIR / "mt5_English_to_Ekegusii", "mt5"),
}

_SENTINEL = re.compile(r"<extra_id_\d+>")
def _clean(s): return _SENTINEL.sub("", s).strip()

def _usable(path):
    p = str(path)
    return Path(p).exists() or ("/" in p and not p.startswith(str(MODEL_DIR)))

@lru_cache(maxsize=4)
def _load(model_path):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE).eval()
    return tok, model

@torch.no_grad()
def translate(text, tgt, src="English"):
    path, mtype = TARGETS[tgt]
    if not _usable(path):
        return f"(no {tgt} model at {path} — check the shared folder / SHARED_FOLDER_URL)"
    tok, model = _load(str(path))
    if mtype == "nllb":
        tok.src_lang = NLLB_CODE[src]
        enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
        gen = model.generate(**enc, max_length=MAX_LEN, num_beams=NUM_BEAMS, no_repeat_ngram_size=3,
                             forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]))
    else:  # mt5
        enc = tok(f"translate {src} to {tgt}: {text}", return_tensors="pt",
                  truncation=True, max_length=MAX_LEN).to(DEVICE)
        gen = model.generate(**enc, max_length=MAX_LEN, num_beams=NUM_BEAMS, no_repeat_ngram_size=3)
    return _clean(tok.batch_decode(gen, skip_special_tokens=True)[0])

print("Ready. Targets:", list(TARGETS.keys()))

## 4 · Translate one sentence you type in

In [ ]:
text = input("Enter an English PSA to translate: ").strip()
if not text:
    print("No text entered.")
else:
    print("\n" + "=" * 72)
    print("EN :", text)
    for tgt in TARGETS:
        print(f"{tgt[:2].upper()} :", translate(text, tgt))

## 5 · Keep translating (type `quit` to stop)

In [ ]:
print("Type an English PSA and press Enter (or 'quit' to stop).\n")
while True:
    text = input("EN > ").strip()
    if text.lower() in ("quit", "exit", "q", ""):
        print("Done."); break
    for tgt in TARGETS:
        print(f"  {tgt}: {translate(text, tgt)}")
    print()